# Importing libraries and downloading the data

In [3]:
import pandas as pd
import numpy as np
import sklearn as sk
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import math


In [5]:
from google.colab import files
uploaded = files.upload() #Make sure to upload 'Student_Exam_Performance.csv'

Saving Student_Exam_Performance.csv to Student_Exam_Performance (1).csv


In [6]:
student_exams_df = pd.read_csv("Student_Exam_Performance.csv")
student_exams_df.head()

,race/ethnicity,parental level of education,lunch,test preparation course,math percentage,reading score percentage,writing score percentage,sex
0,group B,bachelor's degree,standard,none,0.72,0.72,0.74,F
1,group C,some college,standard,completed,0.69,0.90,0.88,F
2,group B,master's degree,standard,none,0.90,0.95,0.93,F
3,group A,associate's degree,free/reduced,none,0.47,0.57,0.44,M
4,group C,some college,standard,none,0.76,0.78,0.75,M


In [7]:
student_exams_df.describe()

,math percentage,reading score percentage,writing score percentage
count,1000.000000,1000.000000,1000.000000
mean,0.660890,0.691690,0.680540
std,0.151631,0.146002,0.151957
min,0.000000,0.170000,0.100000
25%,0.570000,0.590000,0.577500
50%,0.660000,0.700000,0.690000
75%,0.770000,0.790000,0.790000
max,1.000000,1.000000,1.000000


#Initial Data Manipulation
This will make the rest of the project a bit easier to do.

In [8]:
student_exams_df.columns = [c.replace(' ', '_') for c in 
                            student_exams_df.columns]

for col in student_exams_df.columns:
    print(col)

#Making the column names a single word so that it can be called later when
#it is needed for grouping.

race/ethnicity
parental_level_of_education
lunch
test_preparation_course
math_percentage
reading_score_percentage
writing_score_percentage
sex


In [9]:
total_score = (student_exams_df.math_percentage + 
               student_exams_df.reading_score_percentage + 
               student_exams_df.writing_score_percentage)/3

student_exams_df["combined_score"] = total_score
student_exams_df.describe()

,math_percentage,reading_score_percentage,writing_score_percentage,combined_score
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.660890,0.691690,0.680540,0.677707
std,0.151631,0.146002,0.151957,0.142573
min,0.000000,0.170000,0.100000,0.090000
25%,0.570000,0.590000,0.577500,0.583333
50%,0.660000,0.700000,0.690000,0.683333
75%,0.770000,0.790000,0.790000,0.776667
max,1.000000,1.000000,1.000000,1.000000


We added a total score column to input two variables into the t-test: the independent variable (race/ethnicity, parental_level_of_education, lunch, and test_preparation_course) and the combined score for each student.

In [10]:
low_score_df = student_exams_df.loc[(student_exams_df.combined_score < 
                                     0.583333)]

low_score_df.head()

#Using the .loc function to create a dataset with just students who scored 
#below the 25th percentile in all three subjects (math, reading, and writing)

,race/ethnicity,parental_level_of_education,lunch,test_preparation_course,math_percentage,reading_score_percentage,writing_score_percentage,sex,combined_score
3,group A,associate's degree,free/reduced,none,0.47,0.57,0.44,M,0.493333
7,group B,some college,free/reduced,none,0.40,0.43,0.39,M,0.406667
9,group B,high school,free/reduced,none,0.38,0.60,0.50,F,0.493333
10,group C,associate's degree,standard,none,0.58,0.54,0.52,M,0.546667
11,group D,associate's degree,standard,none,0.40,0.52,0.43,M,0.450000


Creating a new dataset allows us to see what are some common factors amongst low scoring students to speculate variables that influence score.

Looking at these values, let us create subgroups for the 5 independent
variables (race/ethnicity, parental level of education, lunch, and test 
preparation courses, and sex) to see if certain variables affect test scores. This will be done by doing t-tests for the independent variables to see how statistically significant they differ from the other values.

# Race/Ethnicity Analysis

In [11]:
race_group = student_exams_df.groupby(["race/ethnicity"])
race_group["math_percentage", "reading_score_percentage", 
           "writing_score_percentage", "combined_score"].median()

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:2: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  


,math_percentage,reading_score_percentage,writing_score_percentage,combined_score
race/ethnicity,,,,
group A,0.610,0.64,0.62,0.613333
group B,0.630,0.67,0.67,0.650000
group C,0.650,0.71,0.68,0.683333
group D,0.690,0.71,0.72,0.700000
group E,0.745,0.74,0.72,0.735000


From this, we can see that there is a discrepancy between the races, with  race A having the lowest scores and race E having the highest. Thus, there  should be an emphasis to improve the scores of students from ethnicities that
come earlier in the alphabet.

Note: I am using median in these models to find the measure of center, as this best compensates for outliers in the group.

In [12]:
stats.ttest_ind(student_exams_df["combined_score"]
                [student_exams_df["race/ethnicity"] == "group C"], #replace 'group C' with the group that you want to test
                student_exams_df["combined_score"]
                [student_exams_df["race/ethnicity"] == "group D"]) #replace 'group D' with the group that you want to test

#Group names: 'group A', 'group B', 'group C', 'group D', 'group E' 

Ttest_indResult(statistic=-1.8063576953601048, pvalue=0.0713815867381346)

These t-tests have shown us that there is a clear divide in performance between certain races, as group E is statistically different than every other group (the p-value for their t-tests are <0.05). D is similar, with a p-value of 0.07 when compared with C and all other values <0.05. This means that D and E are statistically different from the rest of the sets. Thus, the schoolboard should focus on improving the educational quality for students in groups A, B, and C. This is because the t-tests prove that being of race A, B, or C face unique obstacles that affect their scores.

# Parental Education Analysis

In [13]:
education_group = student_exams_df.groupby(["parental_level_of_education"])
education_group["math_percentage", "reading_score_percentage", 
                "writing_score_percentage", "combined_score"].median()

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:2: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  


,math_percentage,reading_score_percentage,writing_score_percentage,combined_score
parental_level_of_education,,,,
associate's degree,0.670,0.725,0.705,0.696667
bachelor's degree,0.680,0.730,0.740,0.711667
high school,0.630,0.660,0.640,0.650000
master's degree,0.730,0.760,0.750,0.733333
some college,0.675,0.705,0.700,0.686667
some high school,0.650,0.670,0.660,0.666667


This dataset shows us that there is also an apparent difference between 
students that have parents who have completed more education (masters or 
bachelor's vs. associate's and high school degrees). Again, similar to the 
race element, it is important to be cognizant of students who have parents with lower educational backgrounds. 

In [14]:
stats.ttest_ind(student_exams_df["combined_score"]
                [student_exams_df["parental_level_of_education"] == 
                 "bachelor's degree"], #replace 'some college' with the education level that you want to test
                student_exams_df["combined_score"]
                [student_exams_df["parental_level_of_education"] == 
                 "some college"]) #replace 'some high school' with the education level that you want to test

#Education Levels: 'master's degree', 'bachelor's degree', 'associate's degree', 
#'some college', 'high school', 'some high school'

Ttest_indResult(statistic=2.200746866670843, pvalue=0.02842185394179664)

These t-tests are able to tell us more about how the college education of their parents affects students and their scores. Mainly, the differential between students whose parents did not go to college at all (high school and some high school) is significantly different from students whose parents went to any college. Another significant difference to note is that students whose parents did not complete a degree (some college), have signfiicantly different test scores than students whose parents have a bachelor's degree or higher. This data tells us that the school board should focus on providing resources to students whose parents did not go to college at all, with leftover funding dedicated to students whose parents did not complete college. 

# Lunch type Analysis

In [15]:
lunch_group = student_exams_df.groupby(["lunch"])
lunch_group["math_percentage", "reading_score_percentage", 
                "writing_score_percentage", "combined_score"].median()

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:2: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  


,math_percentage,reading_score_percentage,writing_score_percentage,combined_score
lunch,,,,
free/reduced,0.60,0.65,0.64,0.626667
standard,0.69,0.72,0.72,0.713333


In [16]:
stats.ttest_ind(student_exams_df["combined_score"]
                [student_exams_df["lunch"] == 
                 "free/reduced"],
                student_exams_df["combined_score"]
                [student_exams_df["lunch"] == 
                 "standard"]) 

Ttest_indResult(statistic=-9.575113051511467, pvalue=7.736791812496048e-21)

This p-test shows that there is a signfiicant difference between students who eat free/reduced lunch and standard lunch. 

# Test Preparation Course Analysis

In [17]:
test_prep_group = student_exams_df.groupby(["test_preparation_course"])
test_prep_group["math_percentage", "reading_score_percentage", 
                "writing_score_percentage", "combined_score"].median()

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:2: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  


,math_percentage,reading_score_percentage,writing_score_percentage,combined_score
test_preparation_course,,,,
completed,0.69,0.75,0.76,0.735000
none,0.64,0.67,0.65,0.653333


In [18]:
stats.ttest_ind(student_exams_df["combined_score"]
                [student_exams_df["test_preparation_course"] == 
                 "completed"],
                student_exams_df["combined_score"]
                [student_exams_df["test_preparation_course"] == 
                 "none"]) 

Ttest_indResult(statistic=8.390944443482587, pvalue=1.6337802035924099e-16)

Proof that taking a test preparation course causes a significant difference in scores between students.

# Student Sex Analysis

In [19]:
sex_group = student_exams_df.groupby(["sex"])
sex_group["math_percentage", "reading_score_percentage", 
                "writing_score_percentage", "combined_score"].median()

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:2: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  


,math_percentage,reading_score_percentage,writing_score_percentage,combined_score
sex,,,,
F,0.65,0.73,0.74,0.703333
M,0.69,0.66,0.64,0.663333


In [20]:
stats.ttest_ind(student_exams_df["math_percentage"]
                [student_exams_df["sex"] == 
                 "F"],
                student_exams_df["math_percentage"]
                [student_exams_df["sex"] == 
                 "M"]) 

Ttest_indResult(statistic=-5.3832458698289845, pvalue=9.120185549328735e-08)

In [21]:
stats.ttest_ind(student_exams_df["reading_score_percentage"]
                [student_exams_df["sex"] == 
                 "F"],
                student_exams_df["reading_score_percentage"]
                [student_exams_df["sex"] == 
                 "M"]) 

Ttest_indResult(statistic=7.959308005187659, pvalue=4.680538743933239e-15)

In [22]:
stats.ttest_ind(student_exams_df["writing_score_percentage"]
                [student_exams_df["sex"] == 
                 "F"],
                student_exams_df["writing_score_percentage"]
                [student_exams_df["sex"] == 
                 "M"]) 

Ttest_indResult(statistic=9.9795579100045, pvalue=2.0198777068680646e-22)

This data statistically proves that there is a significant difference in the scores between women and men. Although men do better in math, women do better in both reading and writing. This is interesting to note, as resources can be focused in a way that allows male students to get more attention in reading/writing, while women get more focus in math.

# Finding Correlation between Independent Variables

Seeing how group A, B, and C are disproportionately affected in terms of race, as well as students whose parents did not go to college, we can find what percentage of students that scored in the lowest quartile (25th percentile) come from these backgrounds.  

In [27]:
lowscore_race_group = low_score_df.groupby(["parental_level_of_education", 
                                            "lunch"])
overall_race_group = student_exams_df.groupby(["parental_level_of_education", 
                                               "lunch"])
(lowscore_race_group["combined_score"].count()/overall_race_group
 ["combined_score"].count())/(9000/2223) 

#the /(9000/2223) is used to normalize the data so that the percents add up to 
#100%

parental_level_of_education  lunch       
associate's degree           free/reduced    0.083403
                             standard        0.045993
bachelor's degree            free/reduced    0.072977
                             standard        0.020027
high school                  free/reduced    0.130557
                             standard        0.062730
master's degree              free/reduced    0.072042
                             standard        0.007057
some college                 free/reduced    0.068785
                             standard        0.033605
some high school             free/reduced    0.137672
                             standard        0.046051
Name: combined_score, dtype: float64

These percents represent what percent of low scoring students exist in each of the categories. The 13% and 13.7% in rows 5 and 11, respectively, indicate that many of the low scoring students have parents did not get any college education and have to buy free/reduced lunch. Moreover, you can see that only 0.007% of students whose parents have a master's degree who buy standard lunch score low on their exams.  

In [28]:
lowscore_race_group = low_score_df.groupby(["parental_level_of_education", 
                                            "test_preparation_course"])
overall_race_group = student_exams_df.groupby(["parental_level_of_education", 
                                               "test_preparation_course"])
(lowscore_race_group["combined_score"].count()/overall_race_group
 ["combined_score"].count())/(9000/2223) 

parental_level_of_education  test_preparation_course
associate's degree           completed                  0.030122
                             none                       0.075864
bachelor's degree            completed                  0.016109
                             none                       0.054889
high school                  completed                  0.061750
                             none                       0.097036
master's degree              completed                  0.024700
                             none                       0.038000
some college                 completed                  0.019247
                             none                       0.059678
some high school             completed                  0.051325
                             none                       0.096863
Name: combined_score, dtype: float64

This chart shows the relative percents of low scoring students' (categorized based on the degree of education that their parents received) test preparation history. For these low scoring students, regardless of the educational history of their parents, it is apparent that more students have not completed test preparation courses compared to those who have. 

In [30]:
lowscore_race_group = low_score_df.groupby(["parental_level_of_education", 
                                            "race/ethnicity"])
overall_race_group = student_exams_df.groupby(["parental_level_of_education", 
                                               "race/ethnicity"])
(lowscore_race_group["combined_score"].count()/overall_race_group
 ["combined_score"].count())/(9000/2223) 

parental_level_of_education  race/ethnicity
associate's degree           group A           0.088214
                             group B           0.054220
                             group C           0.063333
                             group D           0.059280
                             group E           0.044333
bachelor's degree            group A           0.061750
                             group B           0.037050
                             group C           0.043225
                             group D           0.044107
                             group E           0.013722
high school                  group A           0.109778
                             group B           0.092625
                             group C           0.084906
                             group D           0.089818
                             group E           0.056136
master's degree              group A           0.082333
                             group B           0.041167
    

This table plots what percent of students who had low scores (sorted by parental education level) were of what ethnicity/race. Two numbers stick out particularly: 11% of low scoring students have parents who attended some high school and were of group A, and 10.9% of them who had parents who completed high school were also of group A. Group A, according to our previous research, was the most academically disadvantaged group, with students from this background historically scoring the lowest on exams. This means that educational background is linked to race, and this demographic needs particular attention and resources to make up for the resource-inequality that this causes (in terms of their lunch plan and test preparation course access).

In conclusion, it seems as though certian independent variables affect the test scores of students, these being their race/ethnic background, parental education level, lunch plan, and access to test preparation courses. In terms of race, the three most disadvantaged races were groups A, B, and C, with statistically significant differences between these groups and groups D and E. Moreover, students whose parents did not complete any college had a significant difference in test scores compared to those who did. These two factors were correlated, as the plurality amongst students who scores in the lowest quartile were students from group A race and parents who only had a partial or completed high school education. These two factors impacted other financial freedoms, such as improved lunch plan or access to test preparation courses. In terms of a solution for the school board, there should be standard-lunch vouchers that are given to specific unpriveleged students; Lunch provides vital nutrition and keeps students healthy, which allows them to learn better. Moreover, vouchers should also be given for low-income students for test preparation courses, as these would provide low scoring students an opportunity to improve their scores. These benefits should be prioritized for students whose parents do not have a college education, as well as students in groups A, B, and C. These efforts would allow the school to target their efforts towards students who are scoring lower than their potential, providing them the resources to succeed.